# Day 31: Creative Studio — Generate and Understand Images
Run on a GPU with PyTorch, diffusers, transformers, accelerate, Pillow and model weights available. Model downloads may require internet access and acceptance of model terms. CPU generation can be very slow. Outputs are generated locally and were not precomputed here.


## Setup


In [ ]:
import torch, numpy as np
from PIL import Image

# Pick the fastest device available. CPU is fully supported; cuda just makes it faster.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
SEED   = 42  # fix the seed everywhere so results are reproducible and comparable

print(f"device={DEVICE}  dtype={DTYPE}")
print("On CPU keep num_inference_steps small (8-15). On GPU you can raise them freely.")

## Generate a baseline


In [ ]:
from diffusers import StableDiffusionPipeline

# runwayml/stable-diffusion-v1-5 is an open-source (CreativeML OpenRAIL-M) latent diffusion model.
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=DTYPE)
pipe = pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)

STEPS = 12 if DEVICE == "cpu" else 30  # GPU can afford more steps for extra detail
prompt = "a cozy reading nook by a rainy window, warm lamp light, watercolor style"

g = torch.Generator(device=DEVICE).manual_seed(SEED)
img = pipe(prompt, num_inference_steps=STEPS, guidance_scale=7.5, generator=g).images[0]
img.save("gen_baseline.png")
print(f"generated {img.size} image in {STEPS} steps -> gen_baseline.png")
img

## Compare guidance and negative prompts


In [ ]:
def make_grid(images, cols):
    w, h = images[0].size; rows = (len(images)+cols-1)//cols
    grid = Image.new("RGB", (cols*w, rows*h), "white")
    for i, im in enumerate(images):
        grid.paste(im, ((i%cols)*w, (i//cols)*h))
    return grid

variants = [
    ("guidance=3 (loose)",  dict(guidance_scale=3.0,  negative_prompt=None)),
    ("guidance=12 (literal)",dict(guidance_scale=12.0, negative_prompt=None)),
    ("neg: 'blurry, dark'", dict(guidance_scale=7.5, negative_prompt="blurry, dark, lowres, deformed")),
]
imgs = []
for label, kw in variants:
    g = torch.Generator(device=DEVICE).manual_seed(SEED)  # same seed => isolate the knob's effect
    im = pipe(prompt, num_inference_steps=STEPS, generator=g, **kw).images[0]
    imgs.append(im); print("done:", label)

grid = make_grid(imgs, cols=3); grid.save("gen_prompt_grid.png")
print("saved gen_prompt_grid.png"); grid

## Edit with img2img


In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

img2img = StableDiffusionImg2ImgPipeline(**pipe.components).to(DEVICE)
img2img.set_progress_bar_config(disable=True)

init = Image.open("gen_baseline.png").convert("RGB").resize((512, 512))
edit_prompt = "the same cozy reading nook, but at golden sunset, autumn colors"

edits = []
for s in (0.35, 0.65):  # low strength = faithful edit; high = bolder reinterpretation
    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    out = img2img(prompt=edit_prompt, image=init, strength=s,
                  guidance_scale=7.5, num_inference_steps=STEPS, generator=g).images[0]
    edits.append(out); print(f"img2img strength={s} done")

grid2 = make_grid([init] + edits, cols=3); grid2.save("gen_img2img.png")
print("saved gen_img2img.png (original | strength .35 | strength .65)"); grid2

## Caption and classify


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration, CLIPProcessor, CLIPModel

# --- BLIP: open-source image captioning ---
blip_proc  = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(DEVICE)

scene = Image.open("gen_baseline.png").convert("RGB")
inputs = blip_proc(scene, return_tensors="pt").to(DEVICE)
caption = blip_proc.decode(blip_model.generate(**inputs, max_new_tokens=30)[0], skip_special_tokens=True)
print("BLIP caption:", caption)

# --- CLIP: zero-shot classification ---
clip_model = CLIPModel.from_pretrained("laion/CLIP-ViT-B-32-laion2B-s34B-b79K").to(DEVICE)
clip_proc  = CLIPProcessor.from_pretrained("laion/CLIP-ViT-B-32-laion2B-s34B-b79K")

labels = ["a cozy indoor scene", "a busy city street", "a watercolor painting",
          "a photograph of food", "an outdoor landscape"]
ci = clip_proc(text=labels, images=scene, return_tensors="pt", padding=True).to(DEVICE)
with torch.no_grad():
    probs = clip_model(**ci).logits_per_image.softmax(dim=1)[0]
for lab, p in sorted(zip(labels, probs.tolist()), key=lambda t: -t[1]):
    print(f"  {p:5.1%}  {lab}")

## Semantic image search


In [ ]:
import torch.nn.functional as F

gallery_prompts = [
    "a red bicycle leaning on a brick wall",
    "a bowl of ramen with steam rising",
    "a snowy mountain peak at dawn",
    "a cozy reading nook with a lamp",
]
gallery = []
for p in gallery_prompts:
    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    gallery.append(pipe(p, num_inference_steps=STEPS, guidance_scale=7.5, generator=g).images[0])
print(f"built a {len(gallery)}-image gallery")

def clip_image_embeds(images):
    px = clip_proc(images=images, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        return F.normalize(clip_model.get_image_features(**px), dim=-1)

def clip_text_embed(text):
    tx = clip_proc(text=[text], return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        return F.normalize(clip_model.get_text_features(**tx), dim=-1)

img_embeds = clip_image_embeds(gallery)
query = "somewhere warm to sit and read"
scores = (clip_text_embed(query) @ img_embeds.T)[0]
rank = scores.argsort(descending=True)
print(f'\nquery: "{query}"')
for r in rank.tolist():
    print(f"  score={scores[r]:.3f}  <- {gallery_prompts[r]}")

## Provenance and bias probe


In [ ]:
from PIL import PngImagePlugin

# (a) Provenance / watermark metadata — mark AI-generated content so it can be traced downstream.
meta = PngImagePlugin.PngInfo()
meta.add_text("Generator", "stable-diffusion-v1-5 (open-source)")
meta.add_text("AI-Generated", "true")
meta.add_text("Seed", str(SEED))
Image.open("gen_baseline.png").save("gen_baseline_tagged.png", pnginfo=meta)
print("tagged gen_baseline_tagged.png with AI-provenance metadata")
# NOTE: metadata is easily stripped. Robust provenance needs invisible pixel watermarking
# (e.g. the open-source 'invisible-watermark' Stable Diffusion uses) + C2PA content credentials.

# (b) Bias probe — same neutral role prompt, let CLIP read back the tendency.
g = torch.Generator(device=DEVICE).manual_seed(SEED)
role_img = pipe("a photo of a nurse", num_inference_steps=STEPS, generator=g).images[0]
probe = ["a photo of a woman", "a photo of a man"]
pi = clip_proc(text=probe, images=role_img, return_tensors="pt", padding=True).to(DEVICE)
with torch.no_grad():
    pp = clip_model(**pi).logits_per_image.softmax(dim=1)[0]
print("occupation-bias probe ('a nurse'):", {t: f"{p:.1%}" for t, p in zip(probe, pp.tolist())})
print("Training data (LAION web-scrape) skews occupation depictions by gender/ethnicity — observe, don't trust.")

## Check artifacts


In [ ]:
import os
artifacts = ["gen_baseline.png", "gen_prompt_grid.png", "gen_img2img.png",
             "gen_baseline_tagged.png"]
print("Deliverable artifacts:")
for a in artifacts:
    print(f"  {'OK ' if os.path.exists(a) else 'MISSING':3s} {a}  ({os.path.getsize(a) if os.path.exists(a) else 0} bytes)")
print("\nModels used (all open-source):")
print("  - runwayml/stable-diffusion-v1-5   (text-to-image, img2img)")
print("  - Salesforce/blip-image-captioning-base  (captioning)")
print("  - laion/CLIP-ViT-B-32-laion2B-s34B-b79K     (zero-shot classify + image search, local weights)")
print("\nNow write the reflection markdown cell -> see the 5 questions below.")

## Reflection
Compare the generated panels before selecting the best guidance and negative prompt: higher guidance may follow text more closely but can introduce artifacts. Compare both edit strengths to identify the best balance between scene fidelity and the sunset change. Inspect the caption, classification scores and search ranking against the actual images; these scores are relative to the candidate labels and are not calibrated confidence or proof of correctness. Record any mismatches.

PNG metadata records provenance but can be stripped. Add durable content credentials and verify them downstream. A single generated occupation image and binary CLIP labels cannot establish demographic bias: use a diverse prompt set, multiple seeds, suitable measures, human review and careful reporting. Prevent misuse through consent, disclosure, access controls and review. This reflection cannot report observations until the notebook has been run.
